# 🌲 Project 20 — Monitor Deforestation

## 🎯 Goal

Compare vegetation between two different years to identify areas where forest cover has decreased.

We'll use **NDVI** because healthy forests usually have high NDVI values.

---

# 🌍 Why is this useful in Remote Sensing?

Deforestation monitoring is widely used for:

- 🌳 Forest conservation
- 🌍 Climate change research
- 🪵 Illegal logging detection
- 🐘 Wildlife habitat monitoring
- 🌱 Carbon stock estimation

Organizations such as governments and environmental agencies monitor forests regularly using satellite imagery.

---

# 🆕 New Skill: Long-Term Change Detection

Previously (Project 19):

```text
Before Fire
      ↓
NBR
      ↓

After Fire
      ↓
NBR
      ↓

Subtract
      ↓
Burn Map
```

Today:

```text
Year 1
    ↓
NDVI
    ↓

Year 2
    ↓
NDVI
    ↓

Subtract
    ↓
Forest Change
```

Notice that the workflow is almost identical.

The only differences are:

- We compare **years** instead of months.
- We use **NDVI** instead of **NBR**.

---

# 🔗 Connect to Previous Lessons

This project combines ideas you've already learned.

| Project | Skill |
|---------|-------|
| Project 6 | Calculate NDVI |
| Project 7 | Compare NDVI between seasons |
| Project 10 | Change detection |
| Project 16 | Time series |
| Project 20 | Multi-year vegetation change |

You're not learning completely new concepts—you're combining previous skills into a real research workflow.

---

# Step 1 — Define the Study Area

We'll use a forested region in the Amazon rainforest.

```javascript
var studyArea = ee.Geometry.Polygon([
  [
    [-61.0,-4.0],
    [-60.3,-4.0],
    [-60.3,-4.6],
    [-61.0,-4.6],
    [-61.0,-4.0]
  ]
]);
```

This area is in the Amazon rainforest, where forest change can be observed.

---

# Step 2 — Load the Sentinel-2 Collection

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);
```

We'll use the same collection for both years.

---

# Step 3 — Create an NDVI Function

```javascript
function calculateNDVI(image){

  return image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');

}
```

You've written this function several times already.

---

# Step 4 — Select the First Year

We'll create a composite for **2020**.

```javascript
var year2020 = collection
  .filterBounds(studyArea)
  .filterDate('2020-01-01','2020-12-31')
  .median()
  .clip(studyArea);
```

Notice something different.

We use:

```javascript
.median()
```

instead of:

```javascript
.first()
```

### Why?

Because we want one clean image representing the entire year.

This connects directly to **Project 18**.

---

# Step 5 — Select the Second Year

Now create a composite for **2024**.

```javascript
var year2024 = collection
  .filterBounds(studyArea)
  .filterDate('2024-01-01','2024-12-31')
  .median()
  .clip(studyArea);
```

Now we have two annual composite images.

---

# Step 6 — Calculate NDVI

```javascript
var ndvi2020 = calculateNDVI(year2020);

var ndvi2024 = calculateNDVI(year2024);
```

Exactly like previous NDVI projects.

---

# ⭐ Step 7 — Calculate Forest Change

```javascript
var forestChange = ndvi2024.subtract(ndvi2020);
```

Let's understand this.

Suppose one pixel has:

| Year | NDVI |
|------|------|
| 2020 | 0.82 |
| 2024 | 0.35 |

Earth Engine calculates:

```text
0.35 - 0.82 = -0.47
```

A large **negative** value means vegetation decreased, which may indicate deforestation.

If a pixel changes from:

```text
0.45
↓

0.70
```

Then:

```text
0.70 - 0.45 = +0.25
```

Positive values indicate vegetation increased.

---

# Step 8 — Visualization

```javascript
var changeVis = {
  min:-0.5,
  max:0.5,
  palette:[
    'red',
    'white',
    'green'
  ]
};
```

### Interpretation

- 🔴 Red → Vegetation decreased
- ⚪ White → Little or no change
- 🟢 Green → Vegetation increased

---

# Step 9 — Display

```javascript
Map.centerObject(studyArea,10);

Map.addLayer(
  forestChange,
  changeVis,
  'Forest Change'
);
```

Areas in **red** may represent forest loss, while **green** areas indicate vegetation gain or regrowth.

---

# 💻 Complete Code

```javascript
// 1. Study Area
var studyArea = ee.Geometry.Polygon([
  [
    [-61.0,-4.0],
    [-60.3,-4.0],
    [-60.3,-4.6],
    [-61.0,-4.6],
    [-61.0,-4.0]
  ]
]);

// 2. Image Collection
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);

// 3. NDVI Function
function calculateNDVI(image){

  return image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');

}

// 4. Annual Composite (2020)
var year2020 = collection
  .filterBounds(studyArea)
  .filterDate('2020-01-01','2020-12-31')
  .median()
  .clip(studyArea);

// 5. Annual Composite (2024)
var year2024 = collection
  .filterBounds(studyArea)
  .filterDate('2024-01-01','2024-12-31')
  .median()
  .clip(studyArea);

// 6. Calculate NDVI
var ndvi2020 = calculateNDVI(year2020);
var ndvi2024 = calculateNDVI(year2024);

// 7. Forest Change
var forestChange = ndvi2024.subtract(ndvi2020);

// 8. Visualization
var changeVis = {
  min:-0.5,
  max:0.5,
  palette:[
    'red',
    'white',
    'green'
  ]
};

// 9. Display
Map.centerObject(studyArea,10);

Map.addLayer(
  forestChange,
  changeVis,
  'Forest Change'
);
```

---

# 📌 Things to Remember

## 1. Annual Composites

Before comparing years, create a representative image for each year.

```text
2020 Images
      ↓
median()
      ↓
2020 Composite

2024 Images
      ↓
median()
      ↓
2024 Composite
```

This reduces clouds and produces a more reliable comparison.

---

## 2. NDVI Change

The workflow is:

```text
Year 1
   ↓
NDVI
   ↓

Year 2
   ↓
NDVI
   ↓

Subtract
   ↓
Vegetation Change
```

Negative values often indicate vegetation loss.

---

## 3. Interpretation

Remember:

- 🔴 Negative values → Vegetation decreased.
- ⚪ Near zero → Little or no change.
- 🟢 Positive values → Vegetation increased.

> **Important:**  
> A decrease in NDVI may indicate deforestation, but it could also result from drought, wildfire, agriculture, or seasonal differences. Always interpret the results using additional information.

---

# ❌ Common Beginner Mistakes

### ❌ Mistake 1: Comparing single images instead of annual composites.

Single images may contain clouds or seasonal effects.

Annual composites provide a more stable comparison.

---

### ❌ Mistake 2: Comparing different seasons.

For example:

```text
2020 → January

2024 → July
```

Seasonal vegetation differences can be mistaken for deforestation.

Whenever possible, compare the same season or use annual composites.

---

### ❌ Mistake 3: Assuming every NDVI decrease is deforestation.

A lower NDVI does **not** automatically mean trees were cut down.

Other factors such as:

- Fire
- Drought
- Flooding
- Crop harvesting

can also reduce NDVI.

---

# 💻 Your Practice (Project 20.1)

Create a forest change map for a forested region of your choice.

## Requirements

- Create a study area.
- Load the Sentinel-2 collection.
- Create annual composites for two different years using `.median()`.
- Write a `calculateNDVI()` function.
- Calculate NDVI for both years.
- Create a forest change image using:

```javascript
ndviYear2.subtract(ndviYear1)
```

- Display the forest change map.

---

# 🧠 Before Writing the Code

## Question 1

Why do we compare **annual composites** instead of comparing two individual satellite images?

---

## Question 2

A pixel changes from:

```text
2020 NDVI = 0.85
2024 NDVI = 0.40
```

What is the result of:

```javascript
ndvi2024.subtract(ndvi2020)
```

and what might that indicate about the vegetation at that location?